# Planificación de velocidad y trayectoria para un agente autónomo en Tocancipá (Assetto Corsa)
**Física A: Hardware e Información · Proyecto final**

**Integrantes:** John Edward Rincón Aragón (202420895) · Samuel Zuluaga (202520012) · Samuel Beltrán López (202522601)

**Pregunta de investigación.** ¿En qué medida un modelo simplificado de un vehículo permite predecir y optimizar trayectorias de conducción que mantengan un buen rendimiento y estabilidad al ejecutarse en un entorno de simulación con física vehicular más detallada?

**Cómo la respondemos.** Comparamos tres niveles: la estimación matemática, la ejecución con el modelo cinemático de bicicleta y la ejecución en Assetto Corsa, utilizando el mismo agente conductor en los dos últimos. Evaluamos tiempo de vuelta, perfiles de velocidad, desviaciones respecto a la trayectoria objetivo y finalización de ensayos. Comparamos la línea central con la trayectoria optimizada para evaluar el aprovechamiento de la pista.

**Hipótesis.** Un modelo simplificado que incorpora geometría, adherencia, potencia y límites longitudinales puede identificar trayectorias de menor tiempo previsto que la línea central y producir referencias ejecutables en un simulador de física más detallada. La hipótesis se somete a prueba: completar una vuelta no basta si las velocidades o posiciones predichas difieren sustancialmente de las observadas.

## 0. Preparación y fuentes de datos
El notebook analiza archivos guardados, no controla el juego. `experimento_comparativo.py` ejecuta los ensayos; los módulos originales contienen las ecuaciones y el optimizador. Coloque el notebook junto a la carpeta `proy entrega final` o dentro de ella. Seleccione explícitamente la carpeta de una campaña para evitar mezclar ensayos. Si aún no existen resultados, se muestra su estado pendiente, sin sustituir Tocancipá por una pista ficticia.

In [ ]:
import sys, types, json, inspect, csv
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
RAIZ = Path.cwd()
CODIGO = RAIZ / 'proy entrega final' if (RAIZ / 'proy entrega final').is_dir() else RAIZ
sys.path.insert(0, str(CODIGO))
sys.modules.setdefault('pyvjoy', types.ModuleType('pyvjoy'))  # análisis sin mandos
import modelado_fisico as mf
import estimaciones as est
import conducir_fisica_optimizado as cfo
from agente_conductor import AgenteConductor
G = mf.G
plt.rcParams.update({'figure.figsize':(10,4.5),'axes.grid':True})
# Cambiar por la ruta de la campaña que se desea analizar.
EXPERIMENTO = CODIGO / 'experimentos' / 'SELECCIONAR_CAMPAÑA'
PARAM = json.loads((EXPERIMENTO/'parametros.json').read_text(encoding='utf-8')) if (EXPERIMENTO/'parametros.json').exists() else None
print('Campaña:', EXPERIMENTO, '| parámetros disponibles:', PARAM is not None)

## 1. Contexto y aprovechamiento físico de la pista
La línea central es una trayectoria admisible, no necesariamente la de menor tiempo. El ancho de la pista es una oportunidad de optimización: modificar el recorrido cambia su longitud y su curvatura, y con ello las velocidades físicamente posibles. El objetivo no es minimizar distancia ni curvatura aisladamente, sino minimizar el tiempo estimado de vuelta dentro de las restricciones físicas y geométricas.

Assetto Corsa se utiliza como referencia de mayor complejidad física, no como sustituto de una prueba con un automóvil real. Tocancipá se representa con los bordes del mod del circuito. La conexión con Ingeniería de Sistemas y Computación consiste en discretizar el problema, buscar soluciones con restricciones, integrar sensores y actuadores y contrastar las predicciones con datos de simulación.

## 2. Modelo físico y supuestos
El vehículo se representa mediante una bicicleta cinemática con una rueda equivalente por eje y una huella rectangular para comprobar restricciones geométricas. La masa, potencia útil, distancia entre ejes, adherencia y límites longitudinales determinan el movimiento y el perfil de velocidad. Se trabaja en dos dimensiones, sin representar independientemente las fuerzas de cada neumático, la suspensión, la transferencia de peso, el deslizamiento, la aerodinámica o el funcionamiento detallado de la caja de cambios. Estas omisiones reducen el costo computacional y pueden producir errores que se estudiarán experimentalmente.

### 2.2 Ecuaciones del modelo

**Estado y controles.** El estado es $(x, y, \psi, v)$: posición del eje trasero, orientación medida desde el eje $+x$ y rapidez. Los controles son el ángulo de las ruedas delanteras $\delta$ y la aceleración longitudinal $a$.

**Bicicleta cinemática.** Sin deslizamiento lateral, cada rueda avanza en la dirección en la que apunta. Por geometría, el eje trasero gira alrededor de un centro a distancia $R = L/\tan\delta$, y la velocidad angular es $v/R$:

$$\dot x = v\cos\psi,\qquad \dot y = v\sin\psi,\qquad \dot\psi = \frac{v}{L}\tan\delta,\qquad \dot v = a$$

Para seguir una curva de curvatura $\kappa = 1/R$ hace falta $\delta = \arctan(L\kappa)$, y el volante gira $\theta_{volante} = r_{dir}\,\delta$.

**Aceleración lateral.** Un cuerpo que recorre una curva de radio $R$ a rapidez $v$ necesita una aceleración centrípeta $a_{lat} = v^2/R = v^2\kappa$. Esa aceleración la produce la fricción entre neumáticos y asfalto.

**Círculo de fricción.** La fuerza de fricción máxima es $\mu m g$, así que la aceleración total, sumando la parte longitudinal (acelerar o frenar) y la lateral (girar), no puede superar $\mu g$:

$$a^2 + a_{lat}^2 \le (\mu g)^2 \quad\Rightarrow\quad a_{disp} = \sqrt{(\mu g)^2 - (v^2\kappa)^2}$$

**Velocidad máxima en curva.** Si toda la fricción se usa para girar, $m v^2/R = \mu m g$:

$$v_{max} = \sqrt{\frac{\mu g}{|\kappa|}} = \sqrt{\mu g R}$$

La masa se cancela: en el modelo, la velocidad en curva no depende de la masa.

**Potencia del motor.** Con potencia útil $P$, la fuerza de tracción es $F = P/v$. La aceleración queda limitada por el motor y por un tope (agarre de tracción):

$$a_{motor} = \min\left(a_{max},\ \frac{P}{m\,v}\right)$$

**Tramo con aceleración constante.** Entre dos puntos separados $\Delta s$, de $v^2 = v_0^2 + 2a\Delta s$ salen la pasada de aceleración y la de frenado, y la distancia de frenado:

$$v_{i+1} = \sqrt{v_i^2 + 2a\,\Delta s_i},\qquad d = \frac{v_0^2 - v_c^2}{2b}$$

**Tiempo de vuelta.** Con aceleración constante en cada tramo, la velocidad media es $(v_i + v_{i+1})/2$:

$$T = \sum_i \frac{2\Delta s_i}{v_i + v_{i+1}}$$

**Integración en el tiempo (Euler explícito).** Con paso $\Delta t$: $\mathbf{X}_{k+1} = \mathbf{X}_k + \dot{\mathbf{X}}_k\,\Delta t$, donde $\mathbf{X} = (x, y, \psi, v)$. Además, $\delta$ no puede cambiar más de $\dot\delta_{max}\Delta t$ por paso ni superar $\delta_{max}$.

### 2.3 Integraciones y discretización
La geometría permite calcular el rumbo a partir de la curvatura, $	heta(s)=\theta_0+\int_0^s\kappa(z)\,dz$, y la posición mediante la integración de las componentes de la tangente. En la conducción se utilizan los puntos del circuito procesados, no una reconstrucción acumulativa que pueda introducir deriva. El perfil de velocidad se obtiene integrando $v\,dv=a\,ds$ por tramos; el tiempo resulta de integrar $ds/v$. El movimiento temporal del vehículo se aproxima mediante Euler explícito. La separación espacial se mide sobre la geometría exportada, no se presupone constante.

### 2.4 Trazabilidad de ecuaciones en el código

In [ ]:
ecuaciones = [
    ("Pista: ψ = ∫κ ds, E = −∫sin ψ ds, N = ∫cos ψ ds", mf.integrar_pista_desde_curvatura_paper),
    ("Normal izquierda y desplazamiento lateral", mf.desplazar_puntos),
    ("Bicicleta: x', y', ψ', v'",              mf.derivadas_bicicleta),
    ("Paso de Euler con límites de dirección",  mf.paso_bicicleta),
    ("δ = arctan(L κ)",                         mf.curvatura_a_direccion),
    ("Curvatura por tres puntos",               mf.curvaturas_de_puntos),
    ("a_lat = v² κ",                            mf.aceleracion_lateral),
    ("Círculo de fricción",                     mf.aceleracion_total_admisible),
    ("a_disp = √((μg)² − a_lat²)",              mf.aceleracion_longitudinal_por_agarre),
    ("v_max = √(μ g / |κ|)",                    mf.velocidad_maxima_curva),
    ("a_motor = min(a_max, P/(m v))",           mf.aceleracion_motor),
    ("v² = v₀² + 2 a Δs",                       mf.acelerar_en_distancia),
    ("d = (v₀² − v_c²)/(2b)",                   mf.distancia_frenado),
    ("Perfil de velocidad (pasadas atrás/adelante)", est.evaluar_trayectoria),
    ("Tiempo de vuelta T = Σ 2Δs/(vᵢ+vᵢ₊₁)",  cfo.tiempo_vuelta_plan),
    ("Huella rectangular",                      est.esquinas_rectangulo),
    ("Optimización de la trayectoria",          cfo.optimizar),
    ("Agente: mandos a partir del plan",        AgenteConductor.actualizar),
]
print(f"{'Ecuación':52s} {'Archivo':32s} Línea")
for nombre, funcion in ecuaciones:
    archivo = Path(inspect.getsourcefile(funcion)).name
    linea = inspect.getsourcelines(funcion)[1]
    print(f"{nombre:52s} {archivo:32s} {linea}")

## 3. Parámetros del vehículo y procedencia
El inventario siguiente corresponde al constructor `Vehiculo` utilizado por el script experimental. Los parámetros efectivos se guardan en `parametros.json` antes de conducir y se mantienen fijos en toda la campaña.

| Parámetro | Procedencia y justificación |
|---|---|
| Masa $m$ | `car.ini`, `TOTALMASS`; relaciona potencia y aceleración. |
| Potencia útil $P$ | Potencia máxima disponible en telemetría (`maxPower`), tratada como potencia útil constante; aproximación del tren motriz. |
| Batalla $L$ | `suspensions.ini`, distancia entre ejes; determina la relación giro–curvatura. |
| Fricción $\mu$ | Mínimo de `DY_REF` delantero y trasero del compuesto activo en `tyres.ini`; referencia lateral, no medición experimental de fricción efectiva. |
| Aceleración longitudinal $a_{max}$ | Media ponderada experimental: 6,3527 m/s² (91 muestras agrupadas) para Porsche GT2 RS con Hypercar Road. |
| Frenado $b$ | Media ponderada experimental: 9,9336 m/s² (48 muestras agrupadas), magnitud positiva de desaceleración. |
| Relación de dirección | `STEER_RATIO` del vehículo en `car.ini`. |
| Longitud y ancho de carrocería | Valores nominales del constructor: 4,20 m y 1,80 m; pendientes de verificar contra el Porsche. |
| Giro máximo de ruedas | Valor nominal: 30°; no se ha validado la conversión de `STEER_LOCK`. |
| Rapidez máxima de giro de ruedas | Valor nominal: 360°/s; no es una medición del actuador. |
| Gravedad $g$ | 9,81 m/s², constante física. |
| Paso de control $\Delta t$ | 0,02 s nominales; intervalo de registro experimental 0,2 s, con CSV crudo por ciclo. |

Los valores de masa, potencia, batalla y fricción específicos de la campaña se muestran en la celda siguiente. No se fijará arbitrariamente $\mu=1,23$ a partir de una prueba longitudinal. Para otros vehículos o compuestos el lector puede utilizar estimaciones idealizadas: revisar la procedencia guardada.

Si se realiza un ajuste manual posterior, debe documentarse el valor inicial, el nuevo valor y su justificación; la campaña ajustada constituye una nueva calibración y requiere validación independiente.

In [ ]:
from dataclasses import asdict
veh = mf.Vehiculo(**PARAM['vehiculo']) if PARAM else mf.Vehiculo(aceleracion_max_m_s2=6.3527, frenado_max_m_s2=9.9336)
print('Parámetros de campaña' if PARAM else 'SOLO DEMOSTRACIÓN NUMÉRICA: parámetros nominales; no son resultados del Porsche')
for nombre, valor in asdict(veh).items(): print(f'{nombre:38s} {valor:.6g}')
if PARAM: print('Procedencia AC:', PARAM.get('lectura_AC', {}))

## 4. Calibración longitudinal experimental
La gráfica de aceleración y frenado muestra que el desempeño no es constante con la velocidad. Para mantener la simplicidad del modelo elegimos una constante representativa, no el máximo observado (que sobreestimaría buena parte del intervalo) ni el mínimo (que subestimaría buena parte del intervalo).

El promedio de las medias de las bandas, ponderado por el número de muestras, es $\bar b=\sum_i N_i b_i/\sum_i N_i$. Este valor resume la capacidad de frenado observada y no constituye un máximo físico certificado. La misma lógica se aplica a la aceleración. El ensayo llega aproximadamente a 121 km/h; usar esos parámetros por encima de esa velocidad es extrapolación. Los cambios de marcha, transitorios y la aerodinámica son posibles explicaciones de las variaciones, no causas demostradas por la gráfica.

In [ ]:
carpetas = sorted((CODIGO / "medicion_ac_vjoy").glob("*/muestras.csv"))
archivo_muestras = CODIGO / "medicion_ac_vjoy" / "20261125_133617" / "muestras.csv"
datos_m = np.genfromtxt(archivo_muestras, delimiter=",", names=True, dtype=None, encoding="utf-8-sig")
v_kmh = datos_m["velocidad_kmh"].astype(float)
a_med = datos_m["aceleracion_firmada_m_s2"].astype(float)
tipo = datos_m["tipo"].astype(str)
acel, fren = tipo == "acelerando", tipo == "frenando"
print(f"Archivo: {archivo_muestras}")
print(f"Muestras totales: {len(tipo)} | acelerando: {acel.sum()} | frenando: {fren.sum()}")
print(f"Aceleración media medida: {a_med[acel].mean():.2f} m/s²  (mediana {np.median(a_med[acel]):.2f})")
print(f"Frenado medio medido:     {-a_med[fren].mean():.2f} m/s²  (mediana {np.median(-a_med[fren]):.2f})")

from leer_parametros_ac import limites_experimentales
a_cal, b_cal = limites_experimentales("jw_porsche_991.2_gt2_rs_us", "Hypercar road (HR)")
print(f"Calibración del programa (promedio ponderado por muestras de cada banda): a_max = {a_cal:.4f} m/s², b = {b_cal:.4f} m/s²")

# Promedio por bandas de 10 km/h, como en medir_aceleracion_frenado_ac.py
bandas = np.arange(0, 130, 10)
def por_banda(mascara, signo):
    centros, medias = [], []
    for lo in bandas:
        sel = mascara & (v_kmh >= lo) & (v_kmh < lo + 10)
        if sel.sum() >= 5:
            centros.append(lo + 5); medias.append(np.mean(signo * a_med[sel]))
    return np.array(centros), np.array(medias)
vb_a, ab = por_banda(acel, 1); vb_f, fb = por_banda(fren, -1)

plt.scatter(v_kmh[acel], a_med[acel], s=12, alpha=0.35, label="muestras acelerando")
plt.scatter(v_kmh[fren], -a_med[fren], s=12, alpha=0.35, label="muestras frenando (magnitud)")
plt.plot(vb_a, ab, "o-", c="C0", label="media por banda de 10 km/h")
plt.plot(vb_f, fb, "o-", c="C1")
plt.axhline(a_cal, c="C0", ls="--", label=f"modelo: a_max = {a_cal} m/s²")
plt.axhline(b_cal, c="C1", ls="--", label=f"modelo: b = {b_cal} m/s²")
plt.xlabel("velocidad [km/h]"); plt.ylabel("magnitud de la aceleración [m/s²]")
plt.title("Aceleración y frenado medidos en Assetto Corsa (recta, pedal a fondo)")
plt.legend(fontsize=8, loc="lower right"); plt.show()

## 5. Geometría de Tocancipá
Los bordes izquierdo y derecho se emparejan para obtener la línea central y el ancho local. Se orienta la geometría según el estado inicial, se suaviza y se prepara un recorrido con márgenes para continuidad. El tiempo puntuado corresponde únicamente a salida–meta. La referencia espacial común de todas las gráficas es la estación acumulada de la línea central; no se comparan distancias propias de trayectorias diferentes como si fueran equivalentes.

In [ ]:
planes = {tipo: np.load(EXPERIMENTO/f'plan_{tipo}.npz') for tipo in ('central','optimizada') if (EXPERIMENTO/f'plan_{tipo}.npz').exists()}
if 'central' in planes:
    p = planes['central']; s = p['sc'] - p['inicio']; mask = (s>=0)&(s<=p['meta']-p['inicio'])
    ds = np.diff(p['sc'][mask]); print(f'Separación media: {ds.mean():.3f} m; desviación: {ds.std():.3f} m')
    print(f'Longitud de vuelta central: {float(p["meta"]-p["inicio"]):.2f} m')
    fig,ax=plt.subplots(); ax.plot(*p['izquierdo'][mask].T,'k-',lw=.7); ax.plot(*p['derecho'][mask].T,'k-',lw=.7); ax.plot(*p['centro'][mask].T,'--',label='Línea central'); ax.set_aspect('equal'); ax.legend(); plt.show()
else: print('Pendiente: exportación de geometría de la campaña.')

## 6. Optimización física de trayectoria y velocidad
### 6.1 Detección de curvas y ápices geométricos
Se calcula la curvatura de tres puntos consecutivos de la línea central. Para identificar curvas se suaviza con una media móvil de 21 muestras. Se agrupan al menos ocho puntos consecutivos con $|\kappa_{suave}|\geq 0,005\ \mathrm{m}^{-1}$ (radio de 200 m o menor). El máximo absoluto de cada grupo define un ápice geométrico de referencia. Se priorizan los picos más pronunciados, con separación mínima de 35 m, a más de 165 m de los extremos de la vuelta y hasta 20 curvas. El signo identifica el lado interior. Este filtro detecta regiones para buscar candidatos; no elimina la curvatura de la evaluación física.

### 6.2 Aprovechamiento del ancho y candidatos
Cada trayectoria se representa como $\mathbf P_i=\mathbf C_i+d_i\mathbf N_i$, con $\mathbf N_i$ normal izquierda. El desplazamiento permitido depende de los bordes locales y de la reserva geométrica para la huella. Para cada curva se prueban 36 combinaciones: tres posiciones longitudinales del ápice (−20, 0, +20 m), tres aproximaciones al interior (85, 95, 99 % del margen), dos al exterior (70, 95 %) y dos semiextensiones de maniobra (85, 130 m).

La maniobra usa cinco controles espaciales relativos al ápice: $[-1,-0,53,0,0,53,1]$ multiplicados por la extensión (es decir, los cinco factores son −1; −0,53; 0; 0,53; 1). Los extremos conservan el desplazamiento anterior; los controles intermedios describen exterior–interior–exterior. Entre controles se interpola con $q(t)=t^3(10-15t+6t^2)$, de primera y segunda derivadas nulas en los extremos. Esto favorece transiciones suaves, sin garantizar por sí solo curvatura global continua cuando se combina con la geometría discreta.

### 6.3 Evaluación de cada candidata
Se descartan propuestas fuera del corredor o con segmentos degenerados. El evaluador calcula curvatura, límite de velocidad por adherencia y un perfil longitudinal mediante pasadas hacia atrás (frenado) y hacia adelante (aceleración), respetando potencia y agarre restante. Las pasadas se reiteran hasta convergencia o el límite de iteraciones. La condición inicial del experimento es reposo en la salida; los márgenes no se puntúan.

$$v_{lim,i}=\sqrt{\frac{\mu g}{|\kappa_i|}},\quad v_i\leq\sqrt{v_{i+1}^2+2b_i\Delta s_i},\quad v_{i+1}\leq\sqrt{v_i^2+2a_i\Delta s_i}$$

Se aplican además los topes numéricos/configurados del evaluador (registrados en `parametros.json`). El tiempo previsto es $T=\sum_i 2\Delta s_i/(v_i+v_{i+1})$, limitado al intervalo salida–meta.

### 6.4 Selección y búsqueda local
Se acepta una candidata si reduce el tiempo de la vuelta más que el umbral de mejora (0,02 s) y pasa la comprobación discreta de las cuatro esquinas de la huella. En cada curva se conserva la mejor candidata válida encontrada. Se realizan tres pasadas sobre las curvas; las mejoras se acumulan. Esto es una búsqueda local discreta, no una demostración de óptimo global. El mejor ápice no necesariamente minimiza distancia ni curvatura: importa el tiempo de toda la vuelta.

### 6.5 Resultado teórico preliminar
En la ejecución comunicada del optimizador se detectaron 12 curvas, se reportaron 1195 evaluaciones físicas (contador que incluye la evaluación base), siete mejoras aceptadas y cero rechazos por huella entre las candidatas sometidas a esa prueba. El desplazamiento lateral máximo respecto al centro fue 4,904 m; esto no es error de seguimiento. Los tiempos previstos fueron 95,516 s (central) y 93,705 s (optimizada), con una ganancia reportada de 1,810 s, aproximadamente 1,90 %. Estos son resultados de planificación, no vueltas ejecutadas. La celda siguiente muestra los planes de la campaña seleccionada para verificar su correspondencia.

In [ ]:
if all(t in planes for t in ('central','optimizada')):
    fig,ax=plt.subplots(figsize=(9,7)); p=planes['central']; mask=(p['sc']>=p['inicio'])&(p['sc']<=p['meta'])
    for borde in ('izquierdo','derecho'): ax.plot(*p[borde][mask].T,'k-',lw=.7)
    ax.plot(*p['centro'][mask].T,'--',label='Línea central')
    o=planes['optimizada']; m=(o['sc']>=o['inicio'])&(o['sc']<=o['meta']); im=ax.scatter(*o['ruta'][m].T,c=o['v'][m]*3.6,s=5,cmap='plasma',label='Trayectoria optimizada'); fig.colorbar(im,ax=ax,label='Velocidad prevista (km/h)')
    decisiones=(PARAM or {}).get('apices_ultima_decision',[])
    if decisiones:
        a=np.array([x['estacion_m'] for x in decisiones]); ax.scatter(np.interp(a,o['sc'],o['ruta'][:,0]),np.interp(a,o['sc'],o['ruta'][:,1]),marker='^',label='Últimas decisiones de ápice')
    ax.set_aspect('equal'); ax.legend(); plt.show()
    tc,to=[float(planes[t]['tiempo']) for t in ('central','optimizada')]; print(f'Centro: {tc:.3f} s | optimizada: {to:.3f} s | mejora: {100*(tc-to)/tc:.2f}%')
else: print('Pendiente: planes de la campaña; no se genera una pista artificial.')

## 7. Conducción y justificación del agente de IA
Un perfil matemático no especifica necesariamente acciones realizables de forma instantánea. Por ejemplo, pasar de −40° a +40° en un instante no representa un actuador con velocidad finita. El agente conductor transforma las referencias de posición, velocidad y dirección en órdenes de acelerador, freno y dirección, empleando la información del estado actual y límites de actuación. Se utiliza el mismo algoritmo de control en la bicicleta y en Assetto Corsa.

El agente constituye un sistema de IA basado en reglas y control, en el sentido de que percibe su entorno, toma decisiones autónomamente en función de un objetivo y actúa para seguirlo. No se afirma que aprenda, use redes neuronales o reoptimice la ruta durante la vuelta. La planificación física calcula las referencias antes de iniciar el ensayo.

| Elemento | Variables |
|---|---|
| Estado | Posición (x,y), orientación, velocidad y giro equivalente de ruedas |
| Percepción | Estado del simulador, progreso y referencias de la trayectoria |
| Decisión | Comparación con la velocidad objetivo y corrección de dirección |
| Actuadores | Acelerador, freno y dirección |

El periodo nominal es de 20 ms. La diferencia entre estimación y bicicleta permite estudiar la ejecución del plan bajo la propia simplificación; la diferencia entre bicicleta y Assetto Corsa permite estudiar la transferencia, aunque no excluye interacciones entre controlador y dinámica.

## 8. Validación numérica independiente
Antes de interpretar el experimento, comprobamos funciones en situaciones con solución analítica. Estas pruebas pueden ejecutarse con parámetros nominales si la campaña todavía no está disponible; no constituyen resultados de vuelta.

In [ ]:
for R in [20, 50, 150]:
    for ds in [0.5, 2.0, 5.0]:
        theta = np.arange(0, 2 * np.pi, ds / R)
        circulo = np.column_stack([R * np.cos(theta), R * np.sin(theta)])
        k = mf.curvaturas_de_puntos(circulo)
        print(f"R = {R:4d} m, Δs = {ds:3.1f} m → κ calculada = {k.mean():.6f} 1/m, exacta 1/R = {1/R:.6f} 1/m, "
              f"error relativo = {abs(k.mean()*R - 1):.1e}")

### 8.2 Frenado en recta y efecto de $\Delta t$

Frenamos desde $v_0 = 100$ km/h con desaceleración constante $b$ usando `paso_bicicleta` y medimos la distancia hasta detenerse. La solución exacta es $d = v_0^2/2b$.

**Predicción antes de ejecutar.** Euler explícito usa la velocidad del **inicio** de cada paso para avanzar la posición. Al frenar, esa velocidad es mayor que la media del paso, así que el método **sobreestima** la distancia. El exceso en cada paso es $\tfrac{1}{2}b\Delta t^2$, y hay $v_0/(b\Delta t)$ pasos, así que el error total es $\approx \tfrac{1}{2}v_0\Delta t$: **proporcional a $\Delta t$** (método de primer orden). Si duplicamos $\Delta t$, el error se duplica.

In [ ]:
def frenar_en_recta(v0, dt):
    e = mf.Estado(0.0, 0.0, 0.0, v0, 0.0)
    while e.velocidad_m_s > 0:
        e = mf.paso_bicicleta(e, -veh.frenado_max_m_s2, 0.0, dt, veh)
    return e.x

v0 = 100 / 3.6
d_exacta = mf.distancia_frenado(v0, 0.0, veh.frenado_max_m_s2)
dts = np.array([0.2, 0.1, 0.05, 0.02, 0.01, 0.005, 0.001])
errores = np.array([frenar_en_recta(v0, dt) - d_exacta for dt in dts])
for dt, e in zip(dts, errores):
    print(f"Δt = {dt:5.3f} s → error = {e:+8.4f} m  (predicción ½ v₀ Δt = {0.5*v0*dt:7.4f} m)")
pendiente = np.polyfit(np.log(dts), np.log(np.abs(errores)), 1)[0]

plt.loglog(dts, np.abs(errores), "o-", label=f"error de Euler (pendiente ajustada {pendiente:.2f})")
plt.loglog(dts, 0.5 * v0 * dts, "--", label="predicción ½ v₀ Δt (pendiente 1)")
plt.axvline(cfo.INTERVALO_CONTROL_S, c="gray", ls=":", label="Δt del programa en vivo (0,02 s)")
plt.xlabel("paso de tiempo Δt [s]"); plt.ylabel("|d_numérica − d_exacta| [m]")
plt.title(f"Frenado desde 100 km/h (d exacta = {d_exacta:.2f} m)"); plt.legend(fontsize=9); plt.show()

### 8.3 Giro con ángulo constante

Con $\delta$ fijo, la bicicleta debe recorrer un círculo de radio $R = L/\tan\delta$. Simulamos 3 vueltas a un círculo de 50 m a 20 m/s.

**Predicción antes de ejecutar.** En cada paso de Euler, la orientación aumenta exactamente $\omega\Delta t$ (con $\omega = v/R$) y el auto avanza un segmento recto de longitud $v\Delta t$ en la dirección que tenía al inicio del paso. Entonces la trayectoria numérica es un **polígono regular** de lado $v\Delta t$: se cierra sobre sí mismo y no se aleja vuelta tras vuelta. Ese polígono está inscrito en un círculo de radio

$$R_{num} = \frac{v\Delta t}{2\sin(\omega\Delta t/2)} \approx R\left(1 + \frac{(\omega\Delta t)^2}{24}\right)$$

así que el error del radio debería ser muy pequeño y proporcional a $\Delta t^2$. Medimos el radio ajustando un círculo por mínimos cuadrados a todos los puntos simulados.

In [ ]:
R_obj, v_giro = 50.0, 20.0
delta = float(mf.curvatura_a_direccion(1 / R_obj, veh))
print(f"Ángulo de ruedas δ = {np.degrees(delta):.2f}° → volante ≈ {np.degrees(mf.giro_ruedas_a_volante(delta, veh)):.0f}° "
      f"| R = L/tan δ = {mf.direccion_a_radio(delta, veh):.2f} m")
print(f"a_lat = v²/R = {v_giro**2/R_obj:.2f} m/s² ≤ μg = {veh.coef_friccion*G:.2f} m/s² → maniobra viable: "
      f"{bool(mf.aceleracion_total_admisible(v_giro**2/R_obj, 0.0, veh))}")

for dt in [0.2, 0.1, 0.05, 0.02, 0.01]:
    e = mf.Estado(0.0, 0.0, 0.0, v_giro, delta)
    xs, ys = [], []
    for _ in range(int(3 * 2 * np.pi * R_obj / v_giro / dt)):
        e = mf.paso_bicicleta(e, 0.0, delta, dt, veh)
        xs.append(e.x); ys.append(e.y)
    xs, ys = np.array(xs), np.array(ys)
    # círculo x² + y² + Dx + Ey + F = 0 ajustado por mínimos cuadrados
    D, E, F = np.linalg.lstsq(np.c_[xs, ys, np.ones_like(xs)], -(xs**2 + ys**2), rcond=None)[0]
    R_num = np.sqrt(D**2 / 4 + E**2 / 4 - F)
    R_pred = v_giro * dt / (2 * np.sin(v_giro / R_obj * dt / 2))
    print(f"Δt = {dt:4.2f} s → radio medido = {R_num:8.4f} m | predicción vΔt/(2 sin(ωΔt/2)) = {R_pred:8.4f} m | error = {R_num - R_obj:+.4f} m")

## 9. Diseño experimental
Se comparan dos trayectorias: central y optimizada. Cada una tiene una estimación matemática, una ejecución determinista de bicicleta con el agente y diez intentos individuales en Assetto Corsa con el mismo algoritmo conductor. Se alternan las trayectorias y se reinicia manualmente la sesión entre intentos, verificando posición, orientación, reposo, vehículo, compuesto, combustible, temperatura y desgaste dentro de las tolerancias implementadas. No se eliminan fallos ni se ajustan parámetros durante la validación. El control y el registro son independientes: 0,02 s nominales y muestras cada 0,2 s, además de telemetría cruda por ciclo.

| Trayectoria | Estimación | Bicicleta + agente | Assetto Corsa + agente |
|---|---:|---:|---:|
| Central | 1 | 1 | 10 |
| Optimizada | 1 | 1 | 10 |

El número de repeticiones es una elección exploratoria para estudiar consistencia, no una garantía de significancia estadística. La tasa de finalización considera todos los intentos. Los tiempos y perfiles agregados de vueltas completadas se identifican como tales; se conserva la telemetría parcial de los fallos. El detector actual de salida usa el punto de referencia del vehículo; no equivale a una certificación continua de las cuatro ruedas dentro de pista.

## 10. Resultados (pendientes de completar)
### 10.1 Gráficas comparativas
Se muestran las imágenes generadas por `experimento_comparativo.py` de la campaña seleccionada. Las series comparativas son estimación, bicicleta y Assetto Corsa (media de vueltas completadas y banda de una desviación estándar muestral cuando hay suficientes observaciones). Los datos no disponibles se dejan pendientes. El error lateral es la proyección del desplazamiento respecto a la trayectoria objetivo sobre su normal izquierda; el signo distingue ambos lados. El mapa por vuelta muestra su magnitud espacial. Las gráficas de tiempos incluyen los intentos fallidos sin asignarles un tiempo ficticio.

In [ ]:
def mostrar(nombre):
    archivo=EXPERIMENTO/nombre
    if archivo.exists(): display(Image(filename=str(archivo)))
    else: print(f'Pendiente: {nombre}')
for tipo in ('central','optimizada'):
    print('TRAYECTORIA:',tipo)
    for sufijo in ('mapa','speed_kmh','lateral_error_m','tiempos'): mostrar(f'{tipo}_{sufijo}.png')

### 10.2 Mapas espaciales de desviación por vuelta

In [ ]:
mapas=sorted(EXPERIMENTO.glob('*_desviacion.png'))
if mapas:
    for archivo in mapas: print(archivo.stem); mostrar(archivo.name)
else: print('Pendiente: mapas espaciales de desviación.')

### 10.3 Resumen cuantitativo
Se calculará el error de velocidad MAE y RMSE sobre una cuadrícula común de estaciones (sin extrapolar los tramos no recorridos); se reportarán también sesgo, error lateral absoluto medio y máximo, y tasa de finalización. Para el tiempo, $E_T=100|T_{pred}-T_{AC}|/T_{AC}$ y la ganancia de optimización $M=100(T_c-T_o)/T_c$. Se distinguen las medias de vueltas completas y la variabilidad entre ensayos.

In [ ]:
def leer_datos(p): return np.atleast_1d(np.genfromtxt(p,delimiter=',',names=True,dtype=float,encoding='utf-8'))
def interpolar(a, campo, grid):
    if len(a)<2: return np.full_like(grid,np.nan)
    x=np.maximum.accumulate(a['distance_m']); k=np.r_[True,np.diff(x)>1e-8]
    return np.interp(grid,x[k],a[campo][k],left=np.nan,right=np.nan)
medias={}
for tipo in ('central','optimizada'):
    if tipo not in planes: continue
    p=planes[tipo]; grid=np.linspace(0,float(p['meta']-p['inicio']),1001); v=np.interp(grid,p['sc']-p['inicio'],p['v']*3.6)
    archivos=sorted(EXPERIMENTO.glob(f'{tipo}_ac_[0-9][0-9].csv')); completos=[]
    print(' ',tipo, '| tiempo previsto:',float(p['tiempo']))
    for archivo in [EXPERIMENTO/f'{tipo}_bicicleta.csv',*archivos]:
        j=archivo.with_suffix('.json')
        if not archivo.exists() or not j.exists(): continue
        estado=json.loads(j.read_text()); a=leer_datos(archivo); t=estado.get('tiempo_s'); ok=estado.get('completada',False)
        error=v-interpolar(a,'speed_kmh',grid); error=error[np.isfinite(error)]; lateral=np.abs(interpolar(a,'lateral_error_m',grid)); lateral=lateral[np.isfinite(lateral)]
        met={'MAE km/h':float(np.mean(abs(error))) if len(error) else None,'RMSE km/h':float(np.sqrt(np.mean(error**2))) if len(error) else None,'sesgo km/h':float(np.mean(error)) if len(error) else None,'desviación media m':float(lateral.mean()) if len(lateral) else None,'desviación máxima m':float(lateral.max()) if len(lateral) else None,'cobertura %':100*len(error)/len(grid)}
        print(archivo.stem,'completa:',ok,'tiempo:',t,'motivo:',estado.get('motivo'),met)
        if '_ac_' in archivo.stem and ok and t is not None: completos.append(t)
    if archivos: print('Finalizadas:',len(completos),'/',len(archivos),'intentos registrados (meta: 10)')
    if completos:
        medias[tipo]=np.mean(completos); print('Media:',medias[tipo],'SD muestral:',np.std(completos,ddof=1) if len(completos)>1 else 'no estimable','Error tiempo %:',100*abs(float(p['tiempo'])-medias[tipo])/medias[tipo])
if len(medias)==2: print('Mejora observada entre medias de vueltas completas (%):',100*(medias['central']-medias['optimizada'])/medias['central'])
if not planes: print('Pendiente: datos experimentales.')

### 10.4 Interpretación de resultados (completar tras los ensayos)
| Pregunta | Evidencia | Resultado |
|---|---|---|
| ¿La optimización reduce el tiempo previsto? | Plan central vs. optimizado | Preliminar: 1,810 s de mejora reportada |
| ¿Se reproduce la mejora en AC? | Tiempos de vuelta por ensayo | Pendiente |
| ¿Son representativas las velocidades previstas? | MAE, RMSE, sesgo y gráfica de velocidad | Pendiente |
| ¿Dónde se desvía el vehículo? | Error lateral con signo y mapas por ensayo | Pendiente |
| ¿El resultado es repetible? | Tasa de finalización y dispersión | Pendiente |

Relacionar los picos de error con sectores del circuito y condiciones de conducción. Si ambas ejecuciones se apartan de la estimación, investigar controlador y método de estimación. Si la bicicleta sigue el plan y AC no, investigar diferencias físicas y de actuación. Una discrepancia no demuestra automáticamente una causa.

## 11. Limitaciones
El modelo cinemático no reproduce derrapes; un único coeficiente de adherencia no representa temperatura, carga ni desgaste; las capacidades longitudinales se calibraron en un rango limitado y se extrapolan; no se ha verificado la dirección nominal ni la huella contra el vehículo concreto. La potencia útil constante y ausencia de arrastre pueden sesgar el perfil. La búsqueda es local y la comprobación geométrica es discreta. La ejecución del agente depende de la dinámica de cada simulador, y Assetto Corsa sigue siendo una simulación. Los ensayos incompletos deben considerarse al interpretar las medias de los completados.

## 12. Conclusiones (pendientes de validación)
La planificación preliminar predice una reducción del tiempo al aprovechar el ancho del circuito. Todavía no se puede afirmar que la mejora se reproduzca en Assetto Corsa ni que las estimaciones sean suficientemente representativas. Completar la conclusión con las diferencias entre los tres niveles, la ganancia observada y los sectores problemáticos, indicando bajo qué condiciones la simplificación resulta útil y dónde requiere mejoras.

## 13. Declaración de uso de IA y referencias
Se utilizó asistencia de IA para estructurar, documentar y revisar el notebook y el protocolo experimental. Completar la declaración con las herramientas y tareas efectivamente utilizadas por los integrantes, según las normas del curso.

Referencias de partida: Kunos Simulazioni, *Assetto Corsa* (2014); mod de Tocancipá utilizado en el proyecto; código fuente y datos experimentales del equipo. Completar las referencias del texto guía y del artículo de geometría empleado en la versión original, verificando autores, título y edición antes de entregar.